# IAM — 03: Custom Roles and Policies

**Persona:** Catalog Administrator

**Goal:** Create a custom `data_reviewer` role, attach a rate-limit policy on the
STAC search endpoint, assign the role to an existing user, and verify the policy is active.

**Prerequisites:**
- A catalog identified by `CATALOG_ID` already exists
- The principal identified by `PRINCIPAL_ID` has already been provisioned (see notebook 01)
- IAM facade refactor DONE 2026-04-14

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`, `CATALOG_ID`, `PRINCIPAL_ID`

In [ ]:
import os
import json
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL     = os.environ["DYNASTORE_BASE_URL"]
ADMIN_TOKEN  = os.environ["DYNASTORE_ADMIN_TOKEN"]
CATALOG_ID   = os.environ["CATALOG_ID"]
PRINCIPAL_ID = os.environ["PRINCIPAL_ID"]

client = httpx.Client(
    base_url=BASE_URL,
    headers={
        "Authorization": f"Bearer {ADMIN_TOKEN}",
        "Content-Type": "application/json",
    },
    timeout=30.0,
)
print(f"Connected to {BASE_URL}")
print(f"Catalog ID   : {CATALOG_ID}")
print(f"Principal ID : {PRINCIPAL_ID}")

In [ ]:
# Create the custom role 'data_reviewer'
suffix = uuid.uuid4().hex[:6]
role_name = f"data-reviewer-{suffix}"

role_payload = {
    "name": role_name,
    "description": (
        "Read-only reviewer: may search and view STAC items but cannot write "
        "or modify catalog resources."
    ),
}

r = client.post("/admin/roles", content=json.dumps(role_payload))
print(r.status_code, r.text[:400])
assert r.status_code == 201, f"Expected 201 creating role, got {r.status_code}: {r.text}"

role_body = r.json()
# Prefer the canonical name returned by the server
role_name = role_body.get("name") or role_body.get("role_name") or role_name
print(f"Role created: {role_name}")

In [ ]:
# Attach a rate-limit policy: the data_reviewer role is limited to
# 100 POST requests per minute on the STAC search endpoint.
policy_payload = {
    "role": role_name,
    "route_regex": f"/stac/catalogs/{CATALOG_ID}/search",
    "method": "POST",
    "rate_limit": 100,
}

r = client.post("/admin/policies", content=json.dumps(policy_payload))
print(r.status_code, r.text[:400])
assert r.status_code == 201, f"Expected 201 creating policy, got {r.status_code}: {r.text}"

policy_body = r.json()
policy_id = policy_body.get("policy_id") or policy_body.get("id")
assert policy_id, f"policy_id missing from policy response: {policy_body}"
print(f"Policy created — policy_id: {policy_id}")

In [ ]:
# Assign the data_reviewer role to the target principal on this catalog
role_grant = {"roles": [role_name]}

r = client.post(
    f"/admin/users/{PRINCIPAL_ID}/roles/catalogs/{CATALOG_ID}",
    content=json.dumps(role_grant),
)
print(r.status_code, r.text[:300])
assert r.status_code in (201, 204), (
    f"Expected 201 or 204 granting role, got {r.status_code}: {r.text}"
)
print(f"Role '{role_name}' granted to {PRINCIPAL_ID} on catalog {CATALOG_ID}")

In [ ]:
# Verify the policy is active on this catalog
r = client.get("/admin/policies", params={"catalog_id": CATALOG_ID})
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200 listing policies, got {r.status_code}: {r.text}"

policies_body = r.json()
policy_list = policies_body if isinstance(policies_body, list) else policies_body.get("policies", [])

policy_ids_found = [
    p.get("policy_id") or p.get("id") for p in policy_list
]
assert str(policy_id) in [str(pid) for pid in policy_ids_found], (
    f"policy_id {policy_id} not found in catalog policy list: {policy_ids_found}"
)
print(f"Policy {policy_id} confirmed active on catalog {CATALOG_ID}")
for p in policy_list:
    print(f"  {p}")

## Edge cases

### Case A — Unanchored vs anchored `route_regex`

The `route_regex` field is matched against the full request path. An unanchored pattern
like `/search` would match any path containing that substring (e.g. `/admin/search`,
unintended routes). Always anchor patterns with the full catalog prefix or use `^`/`$`
anchors to avoid overly-broad matches.

```
# Dangerous — matches any path containing '/search'
route_regex: "/search"

# Safe — matches only the search endpoint for a specific catalog
route_regex: "/stac/catalogs/my-catalog/search"

# Most precise — anchored with ^ and $
route_regex: "^/stac/catalogs/my-catalog/search$"
```

### Case B — Delete role while a user holds it

Deleting a role that is currently assigned to one or more users may either:
- **Cascade** (role assignment also deleted, user loses the role silently), or
- **Reject** with a 409 Conflict if the platform enforces referential integrity.

Behaviour is implementation-specific. Always revoke role grants before deleting a role
in production to avoid silent capability loss.

In [ ]:
# Demonstrate route_regex pitfalls — print example patterns
print("route_regex pattern examples:")
print()

patterns = [
    ("/search",                                      "DANGEROUS — matches any path with /search"),
    (f"/stac/catalogs/{CATALOG_ID}/search",          "SAFE — matches only this catalog's search"),
    (f"^/stac/catalogs/{CATALOG_ID}/search$",        "SAFEST — anchored, exact match"),
    (f"/stac/catalogs/{CATALOG_ID}/(search|items)",  "MULTI — matches search OR items endpoints"),
]
for pattern, note in patterns:
    print(f"  {note}")
    print(f"    route_regex: {pattern!r}")
    print()

In [ ]:
# Attempt to delete the role while PRINCIPAL_ID still holds it.
# Document actual platform behaviour — do not assert a specific status code.
r = client.delete(f"/admin/roles/{role_name}")
print(f"DELETE /admin/roles/{role_name} → {r.status_code}")
print(r.text[:300])

if r.status_code == 204:
    print("Platform cascaded the deletion — role assignment also removed.")
    print("Note: user silently lost capabilities. Revoke grants before role deletion in production.")
    # Role is already gone; skip the teardown re-delete below
    role_name = None
elif r.status_code == 409:
    print("Platform rejected deletion — role is still in use (referential integrity enforced).")
    print("Revoke the role grant first, then retry the deletion.")
else:
    print(f"Unexpected status {r.status_code} — inspect platform logs.")

## Teardown

Remove the policy and the custom role. If the role was already deleted by the edge-case
cell above (cascade), only the policy deletion is needed.

In [ ]:
r = client.delete(f"/admin/policies/{policy_id}")
print(r.status_code, r.text[:200])
assert r.status_code == 204, f"Expected 204 deleting policy, got {r.status_code}: {r.text}"
print(f"Policy {policy_id} deleted.")

if role_name is not None:
    r = client.delete(f"/admin/roles/{role_name}")
    print(r.status_code, r.text[:200])
    assert r.status_code == 204, f"Expected 204 deleting role, got {r.status_code}: {r.text}"
    print(f"Role '{role_name}' deleted.")
else:
    print("Role already deleted by edge-case cascade — skipping.")

client.close()